<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система строк заказов</h2>

----

### Вариант задания 15


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Крутые линии заказов для какой-нибудь таблицы.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public interface ITrackable
{
    string TrackingNumber { get; set; }
    void UpdateTracking(string status);
}

public interface IReturnable
{
    int ReturnWindowDays { get; set; }
    bool RequestReturn(string reason);
}

class PackageSize {
    public decimal Size {get; private set;}
    public decimal X1 { get; set;}
    public decimal Y1 { get; set;}
    public decimal Z1 { get; set;}
    public decimal X2 { get; set;}
    public decimal Y2 { get; set;}
    public decimal Z2 { get; set;}

    public PackageSize(decimal x1, decimal y1, decimal z1,
        decimal x2, decimal y2, decimal z2) {
            X1 = x1;
            Y1 = y1;
            Z1 = z1;
            X2 = x2;
            Y2 = y2;
            Z2 = z2;
            Size = Math.Abs((x2-x1)*(y2-y1)*(z2-z1));
    }
}

class OrderLine {
    public Guid ProductId {get; private set;} = Guid.NewGuid();
    public string ProductName { get; set;}
    public decimal Price {get; protected set;}
    public PackageSize Box {get; private set;} = new PackageSize(1, 1, 1, 2, 2, 2);
    
    public decimal Weight { get; set; }
    public bool IsFragile { get; set; }
    public string Category { get; set; }

    public OrderLine(string productName, decimal price) {
        ProductName = productName;
        Price = price;
        Weight = 1.0M;
        IsFragile = false;
        Category = "General";
    }

    public virtual decimal CalculateTotal() {
        return Price*1.25M*1.01M + 10;
    }

    public virtual void UpdatePrice(decimal newPrice) {
        Price = newPrice;
    }

    public void ChangeSizeBox(PackageSize box) {
        Box = box;
    }

    public virtual decimal GetShippingCost() {
        return Weight * 5.0M;
    }

    public void PrintWarning() {
        if (IsFragile) {
            Console.WriteLine($"Внимание: Товар {ProductName} хрупкий!");
        }
    }

    public string GetProductDetails() {
        return $"ID продукта: {ProductId}\n" +
            $"Название продукта: {ProductName}\n" +
            $"Себестоимость продукта: {Price}\n" +
            $"Размер упаковки для товара: {Box.Size} м^3\n" +
            $"Вес: {Weight} кг, Категория: {Category}\n";
    }
}

class StandardLine : OrderLine {
    public int Units { get; set;}
    
    public string Supplier { get; set; }
    public DateTime RestockDate { get; set; }

    public StandardLine(string productName, decimal price, int units)
        : base(productName, price){
        Units = units;
        Supplier = "Unknown";
        RestockDate = DateTime.Now.AddDays(7);
    }

    public override decimal CalculateTotal() {
        return (Price*1.25M*1.01M + 10)*Units;
    }

    public bool CheckAvailability() {
        return Units > 0;
    }
}

class BulkyStandardLine : StandardLine, ITrackable, IReturnable {
    public decimal HandlingFee { get; set; }
    
    public string TrackingNumber { get; set; }
    public int ReturnWindowDays { get; set; }

    public BulkyStandardLine(string productName, decimal price, int units, decimal handlingFee)
        : base(productName, price, units) {
        HandlingFee = handlingFee;
        TrackingNumber = "-";
        ReturnWindowDays = 14;
    }

    public override decimal CalculateTotal() {
        return base.CalculateTotal() + HandlingFee;
    }

    public override decimal GetShippingCost() {
        return base.GetShippingCost() + 50.0M;
    }

    public void UpdateTracking(string status) {
        Console.WriteLine($"Статус доставки ({TrackingNumber}): {status}");
    }

    public bool RequestReturn(string reason) {
        Console.WriteLine($"Запрос на возврат {ProductName} по причине: {reason}");
        return true;
    }
}

class SpecialLine: OrderLine {
    public int PercentDiscount { get; set;}
    
    public string PromoCode { get; set; }
    public DateTime ExpirationDate { get; set; }

    public SpecialLine(string productName, decimal price, int percentDiscount)
        : base(productName, price){
        PercentDiscount = percentDiscount;
        PromoCode = "NONE";
        ExpirationDate = DateTime.Now.AddMonths(1);
    }

    public override void UpdatePrice(decimal newPrice) {
        Price = newPrice*(1M-PercentDiscount/100M);
    }

    public void ApplyPromo(string code) {
        if (code == PromoCode && DateTime.Now <= ExpirationDate) {
            PercentDiscount += 5;
        }
    }
}

class FreeLine : OrderLine, ITrackable {
    public decimal Prepayment { get; set;}
    
    public string SponsorName { get; set; }
    public string TrackingNumber { get; set; }

    public FreeLine(string productName, decimal price, decimal prepayment)
        : base(productName, price){
        Prepayment = prepayment;
        SponsorName = "Anonymous";
        TrackingNumber = "N/A";
    }

    public override decimal CalculateTotal() {
        return Price*1.25M*1.01M + 10 - Prepayment;
    }
    
    public void GetSponsorInfo() {
        Console.WriteLine($"Спонсор бесплатного товара: {SponsorName}");
    }

    public void UpdateTracking(string status) {
        Console.WriteLine($"Отслеживание ({TrackingNumber}): {status}");
    }
}

class Order {
    private List<OrderLine> lines;
    
    public decimal TotalOrderCost { get; private set; }

    public Order() {
        lines = new List<OrderLine>();
        TotalOrderCost = 0;
    }

    public void AddLine(OrderLine line) {
        lines.Add(line);
        TotalOrderCost += line.CalculateTotal();
        TotalOrderCost += line.GetShippingCost();
    }

    public void PrintReceipt() {
        Console.WriteLine("=== ЧЕК ЗАКАЗА ===");
        foreach (var line in lines) {
            line.PrintWarning();
            Console.WriteLine(line.GetProductDetails());
            Console.WriteLine($"Стоимость строки (с доставкой): {line.CalculateTotal() + line.GetShippingCost()}\n");
        }
        Console.WriteLine($"ИТОГО К ОПЛАТЕ: {TotalOrderCost}");
        Console.WriteLine("==================");
    }
}

StandardLine standardItem = new StandardLine("Катушка медной проволоки", 1000, 3);
standardItem.Weight = 2.5M;

SpecialLine specialItem = new SpecialLine("Бочки с метиламином", 500, 20);
specialItem.UpdatePrice(600);
specialItem.IsFragile = true;

FreeLine freeItem = new FreeLine("Клетка с обезьянами", 150, 120);
freeItem.TrackingNumber = "TRK987654321";
freeItem.SponsorName = "Фонд защиты животных";

BulkyStandardLine bulkyItem = new BulkyStandardLine("Промышленный станок", 50000, 1, 1500);
bulkyItem.Weight = 500M;
bulkyItem.TrackingNumber = "BLK123456789";

PackageSize newBox = new PackageSize(0, 0, 0, 3, 3, 3);
standardItem.ChangeSizeBox(newBox);

Order myOrder = new Order();
myOrder.AddLine(standardItem);
myOrder.AddLine(specialItem);
myOrder.AddLine(freeItem);
myOrder.AddLine(bulkyItem);

myOrder.PrintReceipt();

freeItem.GetSponsorInfo();
freeItem.UpdateTracking("В пути");

bulkyItem.UpdateTracking("Прибыл на склад");
bulkyItem.RequestReturn("Не подошел по размеру");

=== ЧЕК ЗАКАЗА ===
ID продукта: 69739d87-61f5-4cef-b891-362691ccafee
Название продукта: Катушка медной проволоки
Себестоимость продукта: 1000
Размер упаковки для товара: 27 м^3
Вес: 2.5 кг, Категория: General

Стоимость строки (с доставкой): 3830.0000

Внимание: Товар Бочки с метиламином хрупкий!
ID продукта: 7ec27b55-219c-42b0-b1af-0f530531a1e5
Название продукта: Бочки с метиламином
Себестоимость продукта: 480.0
Размер упаковки для товара: 1 м^3
Вес: 1.0 кг, Категория: General

Стоимость строки (с доставкой): 621.00000

ID продукта: cde91669-0200-4b3e-a0b0-ce4bdfa15843
Название продукта: Клетка с обезьянами
Себестоимость продукта: 150
Размер упаковки для товара: 1 м^3
Вес: 1.0 кг, Категория: General

Стоимость строки (с доставкой): 84.3750

ID продукта: e947afad-b2fb-4503-b3ee-101a1bb97506
Название продукта: Промышленный станок
Себестоимость продукта: 50000
Размер упаковки для товара: 1 м^3
Вес: 500 кг, Категория: General

Стоимость строки (с доставкой): 67185.0000

ИТОГО К ОПЛАТЕ: 71